## Simple Gen AI app using Langchain

In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()
os.environ['OPENAI_API_KEY']=os.getenv('OPENAI_API_KEY') 
#Langsmith tracking
os.environ['LANGCHAIN_API_KEY']=os.getenv('LANGCHAIN_API_KEY')
os.environ['LANGCHAIN_PROJECT']=os.getenv('LANGCHAIN_PROJECT')
os.environ['LANGCHAIN_TRACING_V2']="true"

In [9]:
## Data Ingestion -- From the website we need to scrap the data
from langchain_community.document_loaders import WebBaseLoader
from pprint import pprint

In [6]:
loader=WebBaseLoader("https://docs.langchain.com/langsmith/observability")
loader

In [14]:
docs=loader.load()
print(docs)

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'language': 'en'}, page_content="LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewEngineTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics.LangSmith works with many framew

In [17]:
##  divide into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(
  chunk_size=100,
  chunk_overlap=20
) 
chunks=text_splitter.split_documents(docs) 
pprint(len(chunks))

31


In [20]:
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings(
  model="text-embedding-3-large",
  base_url=os.getenv("BASE_URL")
) 

In [33]:
from langchain_community.vectorstores import FAISS 
vectorstoredb=FAISS.from_documents(docs,embeddings)


In [34]:
query="Langsmith has two usage limits: total traces and extended"
result=vectorstoredb.similarity_search(query)
result[0].page_content

"LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewEngineTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics.LangSmith works with many frameworks and providers. Browse available integrations to connect your stack including OpenAI, Anthropic, CrewAI, Vercel AI SDK, Pydantic AI, and more.Get startedCreate an accountSign up at smith.langchain.com (no credit card required).\nYou can log in with Google, GitHub, or email.Create an API keyGo

In [35]:
from langchain_openai import ChatOpenAI 
llm=ChatOpenAI(model="gpt-4o",base_url=os.getenv("BASE_URL"))
print(llm)

metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.11', 'langchain-openai': '1.3.5'}} output_version=None profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True} client=<openai.resources.chat.completions.completions.Completions object at 0x0000020846B37A10> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000020846B36CC0> root_client=<openai.OpenAI object at 0x0000020846B354F0

In [36]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
    Answer the following question based only on the provided context.

    <context>
    {context}
    </context>
    """
)
prompt

ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on the provided context.\n\n    <context>\n    {context}\n    </context>\n    '), additional_kwargs={})])

In [37]:
document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on the provided context.\n\n    <context>\n    {context}\n    </context>\n    '), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.11', 'langchain-openai': '1.3.5'}}, output_version=None, profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'ima

In [42]:
from langchain_core.documents import Document
question="Langsmith has two usage limits: tool traces and extended"
relevant_docs=vectorstoredb.similarity_search(question)
document_chain.invoke({
  "input" :question ,
  "context" : relevant_docs
})

'What features does LangSmith Observability provide for LLM applications?\n\nLangSmith Observability offers full visibility into LLM applications by providing features that include individual trace viewing, production-wide performance metrics, and integration with many frameworks and providers. Users can trace and debug their applications with ease, monitor performance through customizable dashboards and alerts, automate workflows with rules and webhooks, collect user feedback, and use the LangSmith Engine to detect and resolve issues. The platform supports setting up different hosting options and connects with tools like Claude and VSCode for real-time answers.'

In [ ]:
## Retriver (It is a interface it is way to get data from vector store db) 

vectorstoredb

In [47]:
from langchain_classic.chains import create_retrieval_chain 
retriever=vectorstoredb.as_retriever() 
retriver_chain=create_retrieval_chain(retriever,document_chain)

In [51]:
response=retriver_chain.invoke({
  "input" : "what is langsmith?"
})

In [53]:
response

{'input': 'what is langsmith?',
 'context': [Document(id='60b8747d-aadd-4472-b5a4-597f56efef4e', metadata={'source': 'https://docs.langchain.com/langsmith/observability', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'language': 'en'}, page_content="LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewEngineTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from in

In [54]:
response['answer']

'LangSmith Observability provides comprehensive visibility into your LLM application, offering features such as individual trace analysis, production-wide performance metrics, and integration with various frameworks and providers. You can get started by creating an account at smith.langchain.com and setting up an API key. Tracing can be added to your app using environment variables, framework integrations, or the SDK. The platform allows you to view, filter, export, and share traces, build dashboards, set alerts, and automate workflows. It also offers features for detecting and diagnosing issues with the LangSmith Engine, collecting feedback, annotating outputs, and more. You can set up a LangSmith instance in various deployment configurations, including cloud, hybrid, or self-hosted.'